## Main Goal: Learning Google Cloud

#### Creating a classification model using bigqueryML on fraud data.
 - Source Data: A public BQ Table `bigquery-public-data.ml_datasets.ulb_fraud_detection`
 - Export this table into csv and store it into a **cloud storage bucket**
 - Now load this csv into another bigquery table in a dataset under the specified project.
 - Use bigquery sql to explore the data using biquery cliend and magic command %%bigquery
 - Split data into train, test, validation partitions in table itself.
 - Create a logistic regression model using SQL
 - Make batch predictions on validationa and test dataset using SQL.

In [56]:
PROJECT_ID = 'projectmail-431520'
REGION = 'us-central1'
EXPERIMENT = 'first-time'

In [54]:
# Task is to copy the above BQ table into our own BQ TABLE
# First export the source table into a csv and store into the cloud storage bucket.
# then load this csv into the target bigquery table
# There are multiple ways to do this
# for now, we are using the python sdk clinet API for this.

# import the client sdk
# create clients

from google.cloud import bigquery
from google.cloud import storage
from google.cloud import aiplatform

bq = bigquery.Client(project=PROJECT_ID)
gcs = storage.Client(project=PROJECT_ID)

In [3]:
# Create a storage bucket
BUCKET = PROJECT_ID
if not gcs.lookup_bucket(BUCKET):
    gs_bucket = gcs.bucket(BUCKET)
    new_bucket = gcs.create_bucket(gs_bucket, location=REGION)
    print(f'Created Bucket: {gcs.lookup_bucket(BUCKET).name}')
else:
    gs_bucket = gcs.bucket(BUCKET)
    print(f'Bucket already exist: {gs_bucket.name}')

Created Bucket: projectmail-431520


In [7]:
# Source table to export into csv and store into cloud storage bucket
BQ_SOURCE = 'bigquery-public-data.ml_datasets.ulb_fraud_detection'
# Input table reference
source_table = bigquery.TableReference.from_string(BQ_SOURCE)
print(source_table)

# Cloud storage location where to save the csv
bucket_op_file_uri = f'gs://{gs_bucket.name}/fraud.csv' 
print(bucket_op_file_uri)

# Target table, Big Query 
BQ_DATASET = 'fraud'
BQ_TABLE = 'fraud_prepped'
BQ_TARGET = f'{PROJECT_ID}.{BQ_DATASET}.{BQ_TABLE}'
destination_table = bigquery.TableReference.from_string(BQ_TARGET)
print(destination_table)

bigquery-public-data.ml_datasets.ulb_fraud_detection
gs://projectmail-431520/fraud.csv
projectmail-431520.fraud.fraud_prepped


In [9]:
# Export the bigquery table as csv to cloud storage
job = bq.extract_table(source_table, bucket_op_file_uri, location="US")
print(job.result())

ExtractJob<project=projectmail-431520, location=US, id=4bb0ddeb-5613-4b5f-8c8e-dc894481a09e>


In [ ]:
# Target: Now create a dataset in the project in bigquery, a dataset contains multiple tables.
dataset = bigquery.Dataset(f'{PROJECT_ID}.{BQ_DATASET}')
dataset.location = REGION
bq.create_dataset(dataset)

In [18]:

from google.cloud.exceptions import NotFound
try:
    table = bq.get_table(BQ_TARGET)
    if table:
        print(f'The table already exists: {BQ_TARGET}')
except NotFound as error:
    print(f'Creating Table ...')
    job_config = bigquery.LoadJobConfig(
        write_disposition = 'WRITE_TRUNCATE',
        source_format = bigquery.SourceFormat.CSV,
        autodetect = True,
        labels = {'experiment': f'{EXPERIMENT}'}
    )
    job = bq.load_table_from_uri(bucket_op_file_uri, destination_table, job_config = job_config)
    job.result()
    print(f'Finished creating table: {BQ_TARGET}')

Creating Table ...
Finished creating table: projectmail-431520.fraud.fraud_prepped


#### DATA EXPLORATION

In [19]:
# Execute using client
query = f"""
SELECT *
FROM `{BQ_TARGET}` TABLESAMPLE SYSTEM (1 PERCENT)
#LIMIT 5
"""
df = bq.query(query = query).to_dataframe()
print(df['V1'].describe())

/opt/conda/lib/python3.10/site-packages/google/cloud/bigquery/table.py:2309: UserWarning: Unable to represent RANGE schema as struct using pandas ArrowDtype. Using `object` instead. To use ArrowDtype, use pandas >= 1.5 and pyarrow >= 10.0.1.
  warnings.warn(_RANGE_PYARROW_WARNING)
/opt/conda/lib/python3.10/site-packages/google/cloud/bigquery/table.py:2323: UserWarning: Unable to represent RANGE schema as struct using pandas ArrowDtype. Using `object` instead. To use ArrowDtype, use pandas >= 1.5 and pyarrow >= 10.0.1.
  warnings.warn(_RANGE_PYARROW_WARNING)
/opt/conda/lib/python3.10/site-packages/google/cloud/bigquery/table.py:2337: UserWarning: Unable to represent RANGE schema as struct using pandas ArrowDtype. Using `object` instead. To use ArrowDtype, use pandas >= 1.5 and pyarrow >= 10.0.1.
  warnings.warn(_RANGE_PYARROW_WARNING)
I0000 00:00:1726932222.204289    5098 config.cc:230] gRPC experiments enabled: call_status_override_on_cancellation, event_engine_dns, event_engine_listen

count    142404.000000
mean         -0.005191
std           1.956188
min         -46.855047
25%          -0.921476
50%           0.016307
75%           1.313065
max           2.439207
Name: V1, dtype: float64


In [29]:
#%%bigquery magic command
%load_ext google.cloud.bigquery

The google.cloud.bigquery extension is already loaded. To reload it, use:
  %reload_ext google.cloud.bigquery


In [30]:
%%bigquery
SELECT COUNT(*) FROM `projectmail-431520.fraud.fraud_prepped` TABLESAMPLE SYSTEM (1 PERCENT)
#SELECT * FROM `bigquery-public-data.ml_datasets.ulb_fraud_detection` TABLESAMPLE SYSTEM (1 PERCENT)

Query is running:   0%|          |

/opt/conda/lib/python3.10/site-packages/google/cloud/bigquery/table.py:2309: UserWarning: Unable to represent RANGE schema as struct using pandas ArrowDtype. Using `object` instead. To use ArrowDtype, use pandas >= 1.5 and pyarrow >= 10.0.1.
  warnings.warn(_RANGE_PYARROW_WARNING)
/opt/conda/lib/python3.10/site-packages/google/cloud/bigquery/table.py:2323: UserWarning: Unable to represent RANGE schema as struct using pandas ArrowDtype. Using `object` instead. To use ArrowDtype, use pandas >= 1.5 and pyarrow >= 10.0.1.
  warnings.warn(_RANGE_PYARROW_WARNING)
/opt/conda/lib/python3.10/site-packages/google/cloud/bigquery/table.py:2337: UserWarning: Unable to represent RANGE schema as struct using pandas ArrowDtype. Using `object` instead. To use ArrowDtype, use pandas >= 1.5 and pyarrow >= 10.0.1.
  warnings.warn(_RANGE_PYARROW_WARNING)


Downloading:   0%|          |

,f0_
0,142403


In [33]:
%%bigquery
# CREATE another table, with additional columns for splits and unique id
CREATE TABLE IF NOT EXISTS `projectmail-431520.fraud.fraud_data` AS
WITH temp_table AS (SELECT *, generate_uuid() transaction_id from `projectmail-431520.fraud.fraud_prepped`)
SELECT *,
    CASE 
        WHEN MOD(ABS(FARM_FINGERPRINT(transaction_id)),10) <= 7 then "TRAIN"
        WHEN MOD(ABS(FARM_FINGERPRINT(transaction_id)),10) <= 8 then "VALIDATE"
        ELSE "TEST"
    END AS splits
FROM temp_table

Query is running:   0%|          |

""


In [35]:
job = bq.query(
    query = f"""
    WITH
        COUNTS_TABLE AS (
            SELECT splits, Class, COUNT(*) as n 
            FROM `projectmail-431520.fraud.fraud_data`
            GROUP BY splits, Class
        )
    SELECT *,
        SUM(n) OVER() as total,
        SAFE_DIVIDE(n, SUM(n) OVER(PARTITION BY Class)) as n_pct_class,
        SAFE_DIVIDE(n, SUM(n) OVER(PARTITION BY splits)) as n_pct_split,
        SAFE_DIVIDE(SUM(n) OVER(PARTITION BY Class), SUM(n) OVER()) as class_pct_to  
    FROM COUNTS_TABLE
    """
)
job.to_dataframe()

/opt/conda/lib/python3.10/site-packages/google/cloud/bigquery/table.py:2309: UserWarning: Unable to represent RANGE schema as struct using pandas ArrowDtype. Using `object` instead. To use ArrowDtype, use pandas >= 1.5 and pyarrow >= 10.0.1.
  warnings.warn(_RANGE_PYARROW_WARNING)
/opt/conda/lib/python3.10/site-packages/google/cloud/bigquery/table.py:2323: UserWarning: Unable to represent RANGE schema as struct using pandas ArrowDtype. Using `object` instead. To use ArrowDtype, use pandas >= 1.5 and pyarrow >= 10.0.1.
  warnings.warn(_RANGE_PYARROW_WARNING)
/opt/conda/lib/python3.10/site-packages/google/cloud/bigquery/table.py:2337: UserWarning: Unable to represent RANGE schema as struct using pandas ArrowDtype. Using `object` instead. To use ArrowDtype, use pandas >= 1.5 and pyarrow >= 10.0.1.
  warnings.warn(_RANGE_PYARROW_WARNING)


,splits,Class,n,total,n_pct_class,n_pct_split,class_pct_to
0,TEST,1,54,284807,0.109756,0.001892,0.001727
1,TEST,0,28487,284807,0.100195,0.998108,0.998273
2,VALIDATE,1,63,284807,0.128049,0.002225,0.001727
3,VALIDATE,0,28248,284807,0.099355,0.997775,0.998273
4,TRAIN,1,375,284807,0.762195,0.001645,0.001727
5,TRAIN,0,227580,284807,0.800450,0.998355,0.998273


#### MODEL TRAINING

In [36]:
%%bigquery
CREATE OR REPLACE MODEL `projectmail-431520.fraud.bqml_logistic_regression`
OPTIONS(
    MODEL_TYPE = 'LOGISTIC_REG',
    AUTO_CLASS_WEIGHTS = TRUE,
    CALCULATE_P_VALUES = TRUE,
    CATEGORY_ENCODING_METHOD = 'DUMMY_ENCODING',
    input_label_cols = ['Class'],
    data_split_col = 'custom_splits',
    data_split_method = 'CUSTOM',
    MODEL_REGISTRY = 'VERTEX_AI',
    VERTEX_AI_MODEL_ID = 'bqml_run1',
    VERTEX_AI_MODEL_VERSION_ALIASES = ['run1']
)
AS SELECT * EXCEPT(transaction_id, splits), 
        CASE
            WHEN splits = 'TRAIN' THEN FALSE
            ELSE TRUE 
        END AS custom_splits
FROM `projectmail-431520.fraud.fraud_data` 
WHERE splits != 'TEST'

Query is running:   0%|          |

""


In [37]:
history = bq.query(
    query = f"""
        SELECT *
        FROM ML.TRAINING_INFO(MODEL `projectmail-431520.fraud.bqml_logistic_regression`)
        ORDER BY iteration"""
).to_dataframe()
history

/opt/conda/lib/python3.10/site-packages/google/cloud/bigquery/table.py:2309: UserWarning: Unable to represent RANGE schema as struct using pandas ArrowDtype. Using `object` instead. To use ArrowDtype, use pandas >= 1.5 and pyarrow >= 10.0.1.
  warnings.warn(_RANGE_PYARROW_WARNING)
/opt/conda/lib/python3.10/site-packages/google/cloud/bigquery/table.py:2323: UserWarning: Unable to represent RANGE schema as struct using pandas ArrowDtype. Using `object` instead. To use ArrowDtype, use pandas >= 1.5 and pyarrow >= 10.0.1.
  warnings.warn(_RANGE_PYARROW_WARNING)
/opt/conda/lib/python3.10/site-packages/google/cloud/bigquery/table.py:2337: UserWarning: Unable to represent RANGE schema as struct using pandas ArrowDtype. Using `object` instead. To use ArrowDtype, use pandas >= 1.5 and pyarrow >= 10.0.1.
  warnings.warn(_RANGE_PYARROW_WARNING)


,training_run,iteration,loss,eval_loss,learning_rate,duration_ms
0,0,0,0.683298,0.681125,0.2,6866
1,0,1,0.644921,0.642633,0.4,9196
2,0,2,0.577020,0.574539,0.8,8298
3,0,3,0.470194,0.467441,1.6,8807
4,0,4,0.335269,0.332228,3.2,8481
5,0,5,0.217603,0.214289,6.4,9371
6,0,6,0.159123,0.155305,12.8,8840
7,0,7,0.122163,0.118162,25.6,9813


#### EVALUATION

In [38]:
query = f"""
SELECT 'TEST' as SPLIT, * FROM ML.EVALUATE (MODEL `projectmail-431520.fraud.bqml_logistic_regression`,
    (SELECT * FROM `projectmail-431520.fraud.fraud_data` WHERE SPLITS='TEST'))
UNION ALL
SELECT 'VALIDATE' as SPLIT, * FROM ML.EVALUATE (MODEL `projectmail-431520.fraud.bqml_logistic_regression`,
    (SELECT * FROM `projectmail-431520.fraud.fraud_data` WHERE SPLITS='VALIDATE'))
UNION ALL
SELECT 'TRAIN' as SPLIT, * FROM ML.EVALUATE (MODEL `projectmail-431520.fraud.bqml_logistic_regression`,
    (SELECT * FROM `projectmail-431520.fraud.fraud_data` WHERE SPLITS='TRAIN'))
"""
bq.query(query = query).to_dataframe()

/opt/conda/lib/python3.10/site-packages/google/cloud/bigquery/table.py:2309: UserWarning: Unable to represent RANGE schema as struct using pandas ArrowDtype. Using `object` instead. To use ArrowDtype, use pandas >= 1.5 and pyarrow >= 10.0.1.
  warnings.warn(_RANGE_PYARROW_WARNING)
/opt/conda/lib/python3.10/site-packages/google/cloud/bigquery/table.py:2323: UserWarning: Unable to represent RANGE schema as struct using pandas ArrowDtype. Using `object` instead. To use ArrowDtype, use pandas >= 1.5 and pyarrow >= 10.0.1.
  warnings.warn(_RANGE_PYARROW_WARNING)
/opt/conda/lib/python3.10/site-packages/google/cloud/bigquery/table.py:2337: UserWarning: Unable to represent RANGE schema as struct using pandas ArrowDtype. Using `object` instead. To use ArrowDtype, use pandas >= 1.5 and pyarrow >= 10.0.1.
  warnings.warn(_RANGE_PYARROW_WARNING)


,SPLIT,precision,recall,accuracy,f1_score,log_loss,roc_auc
0,VALIDATE,0.095710,0.920635,0.980467,0.173393,0.118162,0.981299
1,TEST,0.081169,0.925926,0.980029,0.149254,0.120511,0.986823
2,TRAIN,0.069791,0.909333,0.979913,0.129633,0.124186,0.986526


#### BATCH PREDICTION

In [43]:
query = f"""
SELECT * FROM ML.PREDICT(MODEL `projectmail-431520.fraud.bqml_logistic_regression`, (
    SELECT * FROM `projectmail-431520.fraud.fraud_data` WHERE SPLITS='TEST'
))
"""
test_preds = bq.query(query= query).to_dataframe()

/opt/conda/lib/python3.10/site-packages/google/cloud/bigquery/table.py:2309: UserWarning: Unable to represent RANGE schema as struct using pandas ArrowDtype. Using `object` instead. To use ArrowDtype, use pandas >= 1.5 and pyarrow >= 10.0.1.
  warnings.warn(_RANGE_PYARROW_WARNING)
/opt/conda/lib/python3.10/site-packages/google/cloud/bigquery/table.py:2323: UserWarning: Unable to represent RANGE schema as struct using pandas ArrowDtype. Using `object` instead. To use ArrowDtype, use pandas >= 1.5 and pyarrow >= 10.0.1.
  warnings.warn(_RANGE_PYARROW_WARNING)
/opt/conda/lib/python3.10/site-packages/google/cloud/bigquery/table.py:2337: UserWarning: Unable to represent RANGE schema as struct using pandas ArrowDtype. Using `object` instead. To use ArrowDtype, use pandas >= 1.5 and pyarrow >= 10.0.1.
  warnings.warn(_RANGE_PYARROW_WARNING)


In [47]:
print(test_preds.columns)
test_preds[['predicted_Class', 'predicted_Class_probs', 'Time', 'V1', 'V28', 'Amount', 'Class', 'transaction_id','splits']]

Index(['predicted_Class', 'predicted_Class_probs', 'Time', 'V1', 'V2', 'V3',
       'V4', 'V5', 'V6', 'V7', 'V8', 'V9', 'V10', 'V11', 'V12', 'V13', 'V14',
       'V15', 'V16', 'V17', 'V18', 'V19', 'V20', 'V21', 'V22', 'V23', 'V24',
       'V25', 'V26', 'V27', 'V28', 'Amount', 'Class', 'transaction_id',
       'splits'],
      dtype='object')


,predicted_Class,predicted_Class_probs,Time,V1,V28,Amount,Class,transaction_id,splits
0,0,"[{'label': 1, 'prob': 0.19905651380010458}, {'...",119045,-1.702212,-0.432659,0.00,0,91c33754-7e94-42d8-8bb0-0ffe8ca14758,TEST
1,1,"[{'label': 1, 'prob': 0.6585903893234515}, {'l...",91032,-2.079142,-0.834734,0.00,0,ffdc7052-4312-4593-a3b2-f731eba36c97,TEST
2,0,"[{'label': 1, 'prob': 0.14466617684675773}, {'...",154027,1.866596,-0.032014,0.00,0,6e0ad542-c618-46c1-a094-450ab925de21,TEST
3,0,"[{'label': 1, 'prob': 0.035728802559991266}, {...",44022,1.171138,0.018359,0.00,0,064776b8-9c2f-4d5b-994d-f0fee40e9f67,TEST
4,1,"[{'label': 1, 'prob': 0.9999999999243931}, {'l...",84204,-0.937843,0.729889,0.00,1,34e372f3-6f72-4cc9-801f-3979bb7b84b9,TEST
...,...,...,...,...,...,...,...,...,...
28536,0,"[{'label': 1, 'prob': 0.12331551097594692}, {'...",41503,-3.845978,0.203660,2302.38,0,74f60e34-cf0e-465b-88ce-cc5ea17eb891,TEST
28537,0,"[{'label': 1, 'prob': 0.07828478763914512}, {'...",54051,0.794182,0.030793,135.93,0,68ff3b5a-d7f7-47b6-af24-6b5cab1e4329,TEST
28538,1,"[{'label': 1, 'prob': 0.9965778453421501}, {'l...",4462,-2.303350,-0.153029,239.93,1,5ea29464-0de6-4d4b-99a1-a6009c5bd338,TEST
28539,0,"[{'label': 1, 'prob': 0.024338330412833425}, {...",38270,-3.106629,-0.026106,239.93,0,155988ba-c8d5-403c-9ede-3e74015dff90,TEST


In [48]:
test_preds['predicted_Class_probs'][0]

array([{'label': 1, 'prob': 0.19905651380010458},
       {'label': 0, 'prob': 0.8009434861998954}], dtype=object)

###Vertex AI Endpoint - Create or Retrieve

#### ENDPOINT CREATION

In [59]:
aiplatform.init(project = PROJECT_ID, location = REGION)

In [70]:
SERIES = "bqml"

In [73]:
try:
    endpoint = aiplatform.Endpoint.list(filter = f"display_name={SERIES}")[0]
    print(f"Endpoint Found: {endpoint.display_name}")
except:
    endpoint = aiplatform.Endpoint.create(
        display_name = f"{SERIES}",
        labels = {'series' : f"{SERIES}"}
    )
    print(f"Endpoint Created: {endpoint.display_name}")    


Endpoint Found: bqml


#### DEPLOY MODEL AT THE ENDPOINT

In [75]:
vertex_model = aiplatform.Model(model_name = f'bqml_run1')
vertex_model.resource_name

'projects/516164903379/locations/us-central1/models/bqml_run1'

In [83]:
endpoint.deploy(
    model = vertex_model,
    deployed_model_display_name = vertex_model.display_name,
    traffic_percentage = 100,
    machine_type = 'n1-standard-2'
)

Deploying Model projects/516164903379/locations/us-central1/models/bqml_run1 to Endpoint : projects/516164903379/locations/us-central1/endpoints/3251072264891793408
Deploy Endpoint model backing LRO: projects/516164903379/locations/us-central1/endpoints/3251072264891793408/operations/6531705071695036416


InvalidArgument: 400 Error occurred in Explanation preprocessing. <class 'ValueError'> NodeDef mentions attr 'debug_name' not in Op<name=VarHandleOp; signature= -> resource:resource; attr=container:string,default=""; attr=shared_name:string,default=""; attr=dtype:type; attr=shape:shape; is_stateful=true>; NodeDef: {{node weights}}. (Check whether your GraphDef-interpreting binary is up to date with your GraphDef-generating binary.).


In [96]:
# There can be more than 1 model deplyed on the endpoint with some traffic percentage for load share.
endpoint.list_models()

[id: "3861670654214406144"
 model: "projects/516164903379/locations/us-central1/models/bqml_run1"
 display_name: "bqml_run1"
 create_time {
   seconds: 1727001925
   nanos: 601657000
 }
 dedicated_resources {
   machine_spec {
     machine_type: "n1-standard-2"
   }
   min_replica_count: 1
   max_replica_count: 1
 }
 enable_access_logging: true
 disable_container_logging: true
 model_version_id: "4"]

#### Predictions With Vertex AI Endpoint

In [103]:
query = f"""
    SELECT * EXCEPT (Class, transaction_id, splits)FROM `projectmail-431520.fraud.fraud_data` WHERE SPLITS='TEST' LIMIT 2
"""
test_data = bq.query(query=query).to_dataframe()


/opt/conda/lib/python3.10/site-packages/google/cloud/bigquery/table.py:2309: UserWarning: Unable to represent RANGE schema as struct using pandas ArrowDtype. Using `object` instead. To use ArrowDtype, use pandas >= 1.5 and pyarrow >= 10.0.1.
  warnings.warn(_RANGE_PYARROW_WARNING)
/opt/conda/lib/python3.10/site-packages/google/cloud/bigquery/table.py:2323: UserWarning: Unable to represent RANGE schema as struct using pandas ArrowDtype. Using `object` instead. To use ArrowDtype, use pandas >= 1.5 and pyarrow >= 10.0.1.
  warnings.warn(_RANGE_PYARROW_WARNING)
/opt/conda/lib/python3.10/site-packages/google/cloud/bigquery/table.py:2337: UserWarning: Unable to represent RANGE schema as struct using pandas ArrowDtype. Using `object` instead. To use ArrowDtype, use pandas >= 1.5 and pyarrow >= 10.0.1.
  warnings.warn(_RANGE_PYARROW_WARNING)


##### prediction using python client

In [97]:

newobs = test_data.to_dict(orient = 'records')
print(newobs[0])
prediction = endpoint.predict(instances = newobs[0:1])
print(prediction)

{'Time': 119045, 'V1': -1.7022121323578798, 'V2': -1.4096411947643, 'V3': 1.78971182373642, 'V4': 0.9905051214699592, 'V5': 0.27305354299439805, 'V6': 0.6010193774775561, 'V7': -0.7054345652703521, 'V8': 0.10736119327890599, 'V9': 2.22675294564096, 'V10': 0.0766400692059833, 'V11': -1.8744988270796101, 'V12': -0.34372803075057295, 'V13': -1.0246081916679899, 'V14': -1.4146259579952902, 'V15': -0.9045090015309281, 'V16': 0.16311285185504099, 'V17': -0.559679673189298, 'V18': 0.44362181048969, 'V19': -0.363985304703728, 'V20': -1.30294387835784, 'V21': -0.16093570621089198, 'V22': 1.14305075893123, 'V23': 1.05356404380309, 'V24': 0.640259565299825, 'V25': -0.14406324301021398, 'V26': -0.590230539837292, 'V27': -0.0514877553981904, 'V28': -0.43265929163040795, 'Amount': 0.0}
Prediction(predictions=[{'Class_values': ['1', '0'], 'Class_probs': [0.199056512862574, 0.800943487137426], 'predicted_Class': ['0']}], deployed_model_id='3861670654214406144', metadata=None, model_version_id='4', mod

##### prediction using REST API

In [99]:
import json
with open(f'request.json','w') as file:
    file.write(json.dumps({"instances": newobs[0:1]}))

In [100]:
prediction = !curl -X POST \
-H "Authorization: Bearer "$(gcloud auth application-default print-access-token) \
-H "Content-Type: application/json; charset=utf-8" \
-d @request.json \
https://{REGION}-aiplatform.googleapis.com/v1/{endpoint.resource_name}:predict

prediction

['  % Total    % Received % Xferd  Average Speed   Time    Time     Time  Current',
 '                                 Dload  Upload   Total   Spent    Left  Speed',
 '',
 '  0     0    0     0    0     0      0      0 --:--:-- --:--:-- --:--:--     0',
 '100  1207    0   408  100   799   6375  12484 --:--:-- --:--:-- --:--:-- 18859',
 '{',
 '  "predictions": [',
 '    {',
 '      "predicted_Class": [',
 '        "0"',
 '      ],',
 '      "Class_probs": [',
 '        0.199056512862574,',
 '        0.800943487137426',
 '      ],',
 '      "Class_values": [',
 '        "1",',
 '        "0"',
 '      ]',
 '    }',
 '  ],',
 '  "deployedModelId": "3861670654214406144",',
 '  "model": "projects/516164903379/locations/us-central1/models/bqml_run1",',
 '  "modelDisplayName": "bqml_run1",',
 '  "modelVersionId": "4"',
 '}']

##### prediction using GCloud CLI

In [101]:
prediction = !gcloud beta ai endpoints predict {endpoint.name.rsplit('/',1)[-1]} --region={REGION} --json-request=request.json
prediction

['Using endpoint [https://us-central1-prediction-aiplatform.googleapis.com/]',
 "[{'Class_probs': [0.199056512862574, 0.800943487137426], 'Class_values': ['1', '0'], 'predicted_Class': ['0']}]"]